In [44]:
import pandas as pd

historical_matches = pd.read_csv("../data/processed/historical_matches.csv")
tournament_type_count = historical_matches['tournament'].value_counts().head(20)

print(tournament_type_count)


tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [45]:
# remove all non-fifa recognized matches 

non_fifa_tournaments = [
    'Viva World Cup', 'FIFI Wild Cup', 'ELF Cup', 'World Unity Cup',
    'Atlantic Heritage Cup', 'Hungary Heritage Cup', 'Benedikt Fontana Cup',
    'Tynwald Hill Tournament', 'Island Games', 'Inter Games', 'Muratti Vase',
    'Coupe de l\'Outre-Mer'
]

historical_matches = historical_matches[~historical_matches['tournament'].isin(non_fifa_tournaments)]
historical_matches = historical_matches[~historical_matches['tournament'].str.contains('CONIFA', case=False)]

print(historical_matches.shape)


(48272, 10)


# Evaluate k factors

Would be higher on more important tournaments, and thus would make certain games have more weight

In [46]:
K_WORLD_CUP = 60
K_CONTINENTAL = 50
K_QUALIFIER = 40
K_OTHER = 30
K_FRIENDLY = 20

world_cup = {"FIFA World Cup"}

continental_and_major = {
    "UEFA Euro",
    "Copa América",
    "African Cup of Nations",
    "AFC Asian Cup",
    "Gold Cup",
    "CONCACAF Championship",
    "CCCF Championship",
    "NAFC Championship",
    "Oceania Nations Cup",
    "Confederations Cup",
    "UEFA Nations League",
    "CONCACAF Nations League",
    "CONMEBOL–UEFA Cup of Champions",
    "Mundialito",
    "British Home Championship",
}

friendlies = {
    "Friendly",
    "FIFA Series",
    "CONCACAF Series",
    "FIFA 75th Anniversary Cup",
    "United Arab Emirates Friendship Tournament",
    "African Friendship Games",
    "Beijing International Friendship Tournament",
    "Guangzhou International Friendship Tournament",
}

def k_factor(tournament):
    if tournament in world_cup:
        return K_WORLD_CUP
    if tournament in continental_and_major:
        return K_CONTINENTAL
    if "qualification" in tournament.lower():
        return K_QUALIFIER
    if tournament in friendlies or "friendly" in tournament.lower():
        return K_FRIENDLY
    return K_OTHER

historical_matches["k_factor"] = historical_matches["tournament"].map(k_factor)


k_summary = (
    historical_matches.groupby(["k_factor", "tournament"])
    .size()
    .reset_index(name="matches")
    .sort_values(["k_factor", "matches"], ascending=[False, False])
)
k_summary.head(30)

,k_factor,tournament,matches
173,60,FIFA World Cup,964
166,50,Copa América,869
159,50,African Cup of Nations,845
172,50,UEFA Nations League,658
160,50,British Home Championship,523
163,50,CONCACAF Nations League,422
158,50,AFC Asian Cup,421
167,50,Gold Cup,420
171,50,UEFA Euro,388
162,50,CONCACAF Championship,169


# Calculate elo gains and losses

new_elo = curr_elo + k * goal_multiplier (actual_score - expected_score)

In [47]:
HOME_ADVANTAGE = 100

def home_advantage(neutral):
    return 0 if neutral == "TRUE" else HOME_ADVANTAGE

def rating_difference(home_elo, away_elo, neutral):
    return home_elo + home_advantage(neutral) - away_elo

def expected_score(home_elo, away_elo, neutral=False):
    dr = rating_difference(home_elo, away_elo, neutral)
    return 1 / (1 + 10 ** (-dr / 400))

def actual_score(home_score, away_score):
    if home_score > away_score:
        return 1.0
    if home_score < away_score:
        return 0.0
    return 0.5

def goal_multiplier(home_score, away_score):
    gd = abs(home_score - away_score)
    if gd <= 1:
        return 1.0
    if gd == 2:
        return 1.5
    return (11 + gd) / 8

# if team A gains elo, team B loses exactly the amount of elo team A won
def elo_delta(home_elo, away_elo, home_score, away_score, k, neutral=False):
    expected = expected_score(home_elo, away_elo, neutral)
    actual = actual_score(home_score, away_score)
    g = goal_multiplier(home_score, away_score)
    return k * g * (actual - expected)

# quick checks
print("equal teams, home:", round(expected_score(1500, 1500, neutral=False), 3))
print("equal teams, neutral:", round(expected_score(1500, 1500, neutral=True), 3))
print("G for 1-0 / 2-0 / 4-0:", goal_multiplier(1, 0), goal_multiplier(2, 0), round(goal_multiplier(4, 0), 3))
print("home 2-0 vs equal, K=20:", round(elo_delta(1500, 1500, 2, 0, k=20, neutral=False), 2))


equal teams, home: 0.64
equal teams, neutral: 0.64
G for 1-0 / 2-0 / 4-0: 1.0 1.5 1.875
home 2-0 vs equal, K=20: 10.8


# Calculate elo from 1872-2026 raw data

Every team starts at 1500, and elo is calculated from every historical match

In [48]:
from collections import defaultdict

START_ELO = 1500

matches = historical_matches.copy()

ratings = defaultdict(lambda: START_ELO)
pre_home_elo = []
pre_away_elo = []

loop_cols = ["home_team", "away_team", "home_score", "away_score", "k_factor", "neutral"]

for row in matches[loop_cols].itertuples(index=False):
    home_elo = ratings[row.home_team]
    away_elo = ratings[row.away_team]
    pre_home_elo.append(home_elo)
    pre_away_elo.append(away_elo)

    delta = elo_delta(
        home_elo,
        away_elo,
        row.home_score,
        row.away_score,
        row.k_factor,
        row.neutral,
    )
    ratings[row.home_team] = home_elo + delta
    ratings[row.away_team] = away_elo - delta

matches["home_elo_pre"] = pre_home_elo
matches["away_elo_pre"] = pre_away_elo
matches["elo_diff"] = matches["home_elo_pre"] - matches["away_elo_pre"]
historical_matches = matches

team_elo = (
    pd.Series(dict(ratings), name="elo")
    .sort_values(ascending=False)
    .rename_axis("team")
    .reset_index()
)
team_elo["elo"] = team_elo["elo"].round(1)

print(f"Matches processed: {len(historical_matches)}")
print(f"Teams rated: {len(team_elo)}")
print(f"Mean Elo: {team_elo['elo'].mean():.1f}")
team_elo.head(20)

Matches processed: 48272
Teams rated: 298
Mean Elo: 1500.0


,team,elo
0,Spain,2217.3
1,Argentina,2171.0
2,France,2161.0
3,England,2084.1
4,Colombia,2055.2
5,Brazil,2049.7
6,Portugal,2037.2
7,Ecuador,2017.6
8,Netherlands,2004.1
9,Japan,1990.3


In [50]:
pre_world_cup_rankings = pd.read_csv("../data/raw/pre_world_cup_2026_fifa_rankings.csv")
pre_world_cup_rankings.head(20)

,Nation,FIFA_2022_ranking,FIFA_2026_rank,rank_change
0,France,3,1,2
1,Spain,10,2,8
2,Argentina,2,3,-1
3,England,5,4,1
4,Portugal,9,5,4
5,Brazil,1,6,-5
6,Netherlands,6,7,-1
7,Morocco,11,8,3
8,Belgium,4,9,-5
9,Germany,14,10,4


Fifa rankings and our elo predictions are correlated, which is expected. In this case, the FIFA rankings act as a validator to our elo calculations, which passed.